In [50]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key ="Enter your api key"
)

response = llm.invoke("The first person to land on moon was ...")

response.content

'The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon\'s surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That\'s one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.'

In [52]:
#web scaping
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(["https://jobs.adidas-group.com/job/Holon-Merchandising-Specialist-HaMe/1279844101/?feedId=301201&utm_source=j2w"])
docs = loader.load()



In [53]:
page_data = docs[0].page_content
page_data

'\n\n\n\n\n\n\n\n\n\n\n\n\nMerchandising Specialist Job Details | adidas\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                    By continuing to use and navigate this website, you are agreeing to the use of cookies.\n                \n            \n\n                Accept\n\n                Close\n\n\n\n\nSkip to main content\n\n\n\n\n\n\n\n\nPrepare for interview\nSearch Jobs\nSEE CATEGORIES\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch by Keyword\n\n\n\n\nSearch by Location\n\n\n\n\n\n\n\n                                \xa0\n                            \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPrepare for interview\nSearch Jobs\nSEE CATEGORIES\n\n\n\n\n\n\n\n\n\nView Profile\n\n\n\n\nEmployee Login\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch by Keyword\n\n\n\n\nSearch by Location\n\n\n\n\n\n\nShow More Options\n\n\n\n\n\nLoading...\n\n\n\n\n\n\n\n                                            Team:\n                                        \n\n\nAll\n\n\n\

In [ ]:
#creating a template
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the careers page of the website.
    Your job is to extract the job postings and return them in JSON format containing following keys: 'role', 'experience', 'skils' and 'description'
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):
    """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})

res.content

'```json\n{\n  "role": "Merchandising Specialist",\n  "experience": "1-3 years",\n  "skills": [\n    "Structured, organized and process orientated",\n    "Ability to work effectively within a team environment and under pressure",\n    "Strong analytical skills",\n    "Attention to detail",\n    "Fluent in English",\n    "Advanced user of MS Office suite of products – esp. Excel, Access, PPT",\n    "Dashboarding/Data Visualization experience a plus"\n  ],\n  "description": "This role is responsible for maximizing the sales, profitability, and brand equity through the management and execution of merchandise reports, in-season article reporting and trading, and pre-season planning support."\n}\n```'

In [55]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_res =json_parser.parse(res.content)

In [56]:
json_res

{'role': 'Merchandising Specialist',
 'experience': '1-3 years',
 'skills': ['Structured, organized and process orientated',
  'Ability to work effectively within a team environment and under pressure',
  'Strong analytical skills',
  'Attention to detail',
  'Fluent in English',
  'Advanced user of MS Office suite of products – esp. Excel, Access, PPT',
  'Dashboarding/Data Visualization experience a plus'],
 'description': 'This role is responsible for maximizing the sales, profitability, and brand equity through the management and execution of merchandise reports, in-season article reporting and trading, and pre-season planning support.'}

In [57]:
type(json_res)

dict

In [58]:
import pandas as pd

df = pd.read_csv("D:\GenAI_projects\Cold_Email_Generator\my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [59]:
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# ensure we have a sentence transformer embedding function available
if 'sentence_transformer_ef' not in globals():
    # prefer an existing SentenceTransformer model if available
    if 'model' in globals():
        sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model=model)
    else:
        # fallback to a default model name
        sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.PersistentClient()  # defaults to in-memory
collection = client.create_collection(
    name="Portfolio",
    embedding_function=sentence_transformer_ef,
    get_or_create=True
)


In [60]:
import uuid

for _, row in df.iterrows():
    collection.add(
        documents=[row["Techstack"]],
        metadatas=[{"links": row["Links"]}],
        ids=[str(uuid.uuid4())]
    )

In [61]:
collection.get()

{'ids': ['a442cb3c-cf25-41a7-8da4-31a9a4687c76',
  '93cdf15b-031c-4e37-9bf2-68d199a89960',
  '915ed72a-9c53-4bdc-92dd-31329ac395ba',
  '059f6f57-f93c-44f4-ab02-7c0b2ee0551f',
  '9fb0ddf5-36de-4a6d-aa02-9d65ffb2b20d',
  '096ee31a-6ad2-46b5-955b-cacc451f6e97',
  '5cb26041-026d-4d0b-9419-7de5e62cb6ba',
  'f203d480-c175-4af7-8900-24c2c8c7c7dd',
  'db7d2195-5fcf-42d1-b8dd-f0d7a738c998',
  '6c3fc705-e306-4f58-9b9b-0cd7c46537cc',
  '2f815bd5-e6b3-4817-8196-5eaee8204672',
  '11a94334-e92b-4691-bedf-4b0e20ee18fe',
  'a7b0b847-a32c-402c-94f7-cf13f1b74c98',
  'fe75ea84-15a8-4360-9b6e-2089853b795e',
  'c9155b67-f4ff-4bfc-954a-e74cf2b46c7e',
  '2a31f601-308d-43eb-943d-a34c6e3e8c55',
  '1acf18e6-3878-4497-a1c8-196f68ae43f2',
  'ce0dcd24-c7f6-441f-9512-8bbfc0c00f0d',
  '31eeb67a-fbc3-40c8-b0bf-7407259191d0',
  '92ab629e-42b2-4b6c-9716-986bf0214d83',
  '0a6c794f-eccd-495d-8d18-79b8ce3f6448',
  '0ab50f3a-d6e0-4956-837a-b442d0b254a2',
  'ebaffedb-4fdb-446b-82ab-2f6833b9a97a',
  '7a4f15af-1a14-4462-9a76-

In [62]:
links = collection.query(query_texts=["EXperience in Python", "Expertise in React"],n_results=2)

In [63]:
links

{'ids': [['e97978da-66d3-49bd-b7eb-e4b05998e64f',
   '266e502b-e0fb-4f20-9a64-40046bf08efe'],
  ['0a6c794f-eccd-495d-8d18-79b8ce3f6448',
   'b071be8b-644f-4853-89de-cfa24269e920']],
 'embeddings': None,
 'documents': [['Machine Learning, Python, TensorFlow',
   'Machine Learning, Python, TensorFlow'],
  ['React, Node.js, MongoDB', 'React, Node.js, MongoDB']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'links': 'https://example.com/ml-python-portfolio'},
   {'links': 'https://example.com/ml-python-portfolio'}],
  [{'links': 'https://example.com/react-portfolio'},
   {'links': 'https://example.com/react-portfolio'}]],
 'distances': [[0.4984424114227295, 0.4984424114227295],
  [0.5654599070549011, 0.5654599070549011]]}

In [64]:
job  = json_res
job['skills']

['Structured, organized and process orientated',
 'Ability to work effectively within a team environment and under pressure',
 'Strong analytical skills',
 'Attention to detail',
 'Fluent in English',
 'Advanced user of MS Office suite of products – esp. Excel, Access, PPT',
 'Dashboarding/Data Visualization experience a plus']

In [65]:
links = collection.query(query_texts=job['skills'],n_results=2).get('metadatas',[])

In [66]:
links

[[{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/android-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/ios-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/angular-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/flutter-portfolio'}]]

In [67]:
job

{'role': 'Merchandising Specialist',
 'experience': '1-3 years',
 'skills': ['Structured, organized and process orientated',
  'Ability to work effectively within a team environment and under pressure',
  'Strong analytical skills',
  'Attention to detail',
  'Fluent in English',
  'Advanced user of MS Office suite of products – esp. Excel, Access, PPT',
  'Dashboarding/Data Visualization experience a plus'],
 'description': 'This role is responsible for maximizing the sales, profitability, and brand equity through the management and execution of merchandise reports, in-season article reporting and trading, and pre-season planning support.'}

In [69]:

prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Nikhil, a business development executive at NG.ai pvt.ltd., NG.ai is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of NG.ai
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase NG.ai's portfolio: {link_list}
        Remember you are Nikhil, BDE at NG.ai. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )


chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhance Merchandising Operations with NG.ai's Automated Solutions

Dear Hiring Manager,

I came across the job description for a Merchandising Specialist at your esteemed organization, and I was impressed by the role's focus on maximizing sales, profitability, and brand equity through efficient merchandise management. As a Business Development Executive at NG.ai, I believe our company's expertise in AI and software consulting can significantly enhance your merchandising operations.

Our team at NG.ai has a proven track record of empowering enterprises with tailored solutions, fostering scalability, process optimization, cost reduction, and heightened overall efficiency. We can help you streamline merchandise reporting, in-season article reporting and trading, and pre-season planning support by leveraging our expertise in data visualization and dashboarding.

To give you a glimpse into our capabilities, I'd like to highlight some of our notable projects:

* Our work in data vis